# Chapter 4 - Training Data and Preprocessing for VLMs

In [ ]:
# Here some installs that we will be using in multiple parts of the chapter
!pip install datasets pillow huggingface_hub requests
!pip install transformers==5.2.0
!apt update && apt install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 38.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,388 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 http://archive.ubuntu.com/ubu

In [ ]:
# Very important to be logged in!
from huggingface_hub import login
login()


## 4.1 Looking at the data
### 4.1.1 Image-Text Datasets

*Looking at LAION dataset*

Remember: some datasets are gated, this means that you first have to go to the dataset page and accept the terms and conditions and then you will be able to browse it.





In [ ]:
# Stream LAION dataset without downloading terabytes using the streaming functionality
from datasets import load_dataset

dataset = load_dataset("laion/relaion2B-en-research-safe", streaming=True)

for example in dataset['train'].take(3):
    print(f"Caption: {example['caption']}")
    print(f"Image URL: {example['url']}")
    print("---")


Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]

Caption: """Brent Payne """"Brent Payne"""" 1999 Self Released Country Nm/Nm Out Of Print Cd"""
Image URL: https://www.picclickimg.com/d/l400/pict/333262346250_/Brent-Payne-Brent-Payne-1999-Self-Released-Country.jpg
---
Caption: Universal Orlando 2012 The Ultimate Guide to the Ultimate Theme Park Adventure
Image URL: https://nationalbookswap.com/pbs/m/42/0942/9781887140942.jpg
---
Caption: Unique 14k Gold Yellow and Blue Diamond Engagement Ring 2.64ct.
Image URL: https://d1251d0o0760fi.cloudfront.net/catalog/product/1/4/14k-gold-diamond-engagement-ring-264-ct-p-64.jpg
---


## 4.2 Building a dataset
### 4.2.2 Data Filtering at Scale

Complementing the snippet in section 4.2.2, here is the full code to read videos from a HF Dataset and push back to another repo the videos that are not static

In [ ]:
import os
import math
import subprocess
import tempfile
import requests
from datasets import Dataset, DatasetDict
from huggingface_hub import HfApi


In [ ]:
# CONFIG, set here your input dataset with videos and your output dataset where you want to push the results
INPUT_DATASET_REPO = "vlmbook/videos"
OUTPUT_DATASET_REPO = "<yourUsername>/filtered-videos"

In [ ]:

# A couple of auxiliar functions to explore the input dataset, find MP4 files and download those to analyze them
def get_mp4_files_in_repo():
   """Check which MP4 files are available in the repository root."""
   api = HfApi()

   try:
       repo_files = api.list_repo_files(INPUT_DATASET_REPO, repo_type="dataset")
       # Filter for MP4 files in root only
       mp4_files = [f for f in repo_files if f.endswith('.mp4') and '/' not in f]

       print(f"Found {len(mp4_files)} MP4 files:")
       for mp4_file in mp4_files:
           print(f"  - {mp4_file}")

       return mp4_files

   except Exception as e:
       print(f"Error accessing repository: {e}")
       return []


def download_video_from_hf(video_filename):
   """Download a specific MP4 file from the HF repository."""
   url = f"https://huggingface.co/datasets/{INPUT_DATASET_REPO}/resolve/main/{video_filename}"

   try:
       response = requests.get(url, stream=True)
       response.raise_for_status()

       # Create temporary file
       temp_file = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False)

       # Download video data
       for chunk in response.iter_content(chunk_size=8192):
           temp_file.write(chunk)

       temp_file.close()
       print(f"Downloaded {video_filename}")
       return temp_file.name

   except Exception as e:
       print(f"Error downloading {video_filename}: {e}")
       return None



In [ ]:


# The magic of the script - how we use ffmpeg and its plugin freezedetect to discard videos quickly

def is_video_static(video_file, threshold=0.4):
   """Check if video has static content using ffmpeg freezedetect."""

   # Get video duration
   result = subprocess.run([
       "ffprobe", "-v", "quiet", "-show_entries", "format=duration",
       "-of", "csv=p=0", video_file
   ], capture_output=True, text=True)

   duration = float(result.stdout.strip())
   segments = math.ceil(duration / 60)  # 60-second segments
   freeze_count = 0

   # Check each segment for freezes
   for start in range(0, int(duration), 60):
       result = subprocess.run([
           "ffmpeg", "-ss", str(start), "-i", video_file, "-t", "60",
           "-vf", "freezedetect=n=0.05:d=50", "-f", "null", "-"
       ], capture_output=True, text=True)

       if "freezedetect" in result.stderr:
           freeze_count += 1

   freeze_percentage = freeze_count / segments
   print(f"  Freeze percentage: {freeze_percentage:.1%}")

   return freeze_percentage >= threshold



In [ ]:

# Discover MP4 files in the repository
print("Checking for MP4 files in repository...")
mp4_files = get_mp4_files_in_repo()


# Process each MP4 file
filtered_videos = []


for mp4_file in mp4_files:
   print(f"\nProcessing {mp4_file}...")

   # Extract video ID from filename
   video_id = mp4_file.replace('.mp4', '')

   # Download video from HF repo
   temp_video_path = download_video_from_hf(mp4_file)
   if not temp_video_path:
       continue

   # Analyze video for static content
   try:
       if not is_video_static(temp_video_path):
           # Video passes filter - read file data
           with open(temp_video_path, 'rb') as f:
               video_bytes = f.read()

           filtered_videos.append({
               'video_id': video_id,
               'filename': mp4_file,
               'video_data': video_bytes
           })
           print(f"{video_id} passed filter")
       else:
           print(f"{video_id} filtered out")

   except Exception as e:
       print(f"Error analyzing {video_id}: {e}")

   finally:
       # Clean up temporary file
       if os.path.exists(temp_video_path):
           os.unlink(temp_video_path)


Checking for MP4 files in repository...
Found 4 MP4 files:
  - 09KmKSz4r_Y.mp4
  - MewNUHRGOm0.mp4
  - StKxb6z-MHk.mp4
  - solid_rgb_3min_640x480.mp4

Processing 09KmKSz4r_Y.mp4...
Downloaded 09KmKSz4r_Y.mp4
  Freeze percentage: 0.0%
09KmKSz4r_Y passed filter

Processing MewNUHRGOm0.mp4...
Downloaded MewNUHRGOm0.mp4
  Freeze percentage: 0.0%
MewNUHRGOm0 passed filter

Processing StKxb6z-MHk.mp4...
Downloaded StKxb6z-MHk.mp4
  Freeze percentage: 0.0%
StKxb6z-MHk passed filter

Processing solid_rgb_3min_640x480.mp4...
Downloaded solid_rgb_3min_640x480.mp4
  Freeze percentage: 100.0%
solid_rgb_3min_640x480 filtered out


In [ ]:


# Save filtered results
if filtered_videos:
   print(f"\nSaving {len(filtered_videos)} filtered videos...")

   # Create dataset from filtered videos
   filtered_dataset = Dataset.from_list(filtered_videos)

   # Push filtered videos to new repository
   if OUTPUT_DATASET_REPO is not None:
     filtered_dataset.push_to_hub(f"{OUTPUT_DATASET_REPO}")

     print(f"\nFiltered dataset available at: {OUTPUT_DATASET_REPO}")

   print("Filtered videos saved successfully!")

else:

   print("No videos passed the filter.")


Saving 3 filtered videos...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########| 10.6MB / 10.6MB            

README.md:   0%|          | 0.00/354 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.



Filtered dataset available at: mfarre/testing
Filtered videos saved successfully!


### 4.2.4 Build your own synthesis pipeline
Below the full code to synthesize Q&A using Apple's FastVLM. Note that we use the 1.5B model to make it easier to run in a colab but expect better Q&A pairs with the 7B.

In [ ]:
# Import libraries
import torch
import json
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset
from huggingface_hub import login
import requests
from io import BytesIO

In [ ]:
# Load the model from the hub
# Note: we are loading the 1.5B because it fits in a free Google Colab but for
# better quality Q&A you can use the 7B

model_id = "apple/FastVLM-1.5B"
#model_id = "apple/FastVLM-7B"

IMAGE_TOKEN_INDEX = -200

print("Loading FastVLM model...")
tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)



Loading FastVLM model...


Loading weights:   0%|          | 0/972 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:

# Given an image, FastVLM returns us three Q&As for our dataset

def generate_qa_pairs(image_url):
    try:
        # Download image
        response = requests.get(image_url, timeout=10)
        img = Image.open(BytesIO(response.content)).convert("RGB")

        # Build chat template for Q&A generation
        messages = [
            {"role": "user", "content": "<image>\nGenerate 3 question-answer pairs about this image. Format as JSON: [{\"question\": \"...\", \"answer\": \"...\"}]"}
        ]
        rendered = tok.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        pre, post = rendered.split("<image>", 1)

        # Tokenize around image token
        pre_ids = tok(pre, return_tensors="pt", add_special_tokens=False).input_ids
        post_ids = tok(post, return_tensors="pt", add_special_tokens=False).input_ids
        img_tok = torch.tensor([[IMAGE_TOKEN_INDEX]], dtype=pre_ids.dtype)
        input_ids = torch.cat([pre_ids, img_tok, post_ids], dim=1).to(model.device)
        attention_mask = torch.ones_like(input_ids, device=model.device)

        # Process image
        px = model.get_vision_tower().image_processor(images=img, return_tensors="pt")["pixel_values"]
        px = px.to(model.device, dtype=model.dtype)

        # Generate Q&A pairs
        out = model.generate(
            inputs=input_ids,
            attention_mask=attention_mask,
            images=px,
            max_new_tokens=256,
        )

        result = tok.decode(out[0], skip_special_tokens=True)
        # Extract JSON part after the prompt
        qa_text = result.split("Format as JSON:")[-1].strip()
        return qa_text

    except Exception as e:
        return f"Error: {str(e)}"



In [ ]:
# Here you do three things:
# 1- Stream the LAION dataset (limited to 5 samples in this example)
# 2- Get Q&A pairs from Fast VLM
# 3- Push the results to your own repo

#OUTPUT_REPO = "<yourUsername>/laion-vlm-analysis" # If set to None, it skips the push
OUTPUT_REPO = None
print("Streaming LAION dataset...")
dataset = load_dataset("laion/relaion2B-en-research-safe", streaming=True)

analyzed_examples = []

for i, example in enumerate(dataset['train'].take(5)):
    print(f"\nExample {i+1}:")
    print(f"Original caption: {example['caption']}")
    print(f"Image URL: {example['url']}")

    # Generate Q&A pairs with VLM
    qa_pairs = generate_qa_pairs(example['url'])
    print(f"Generated Q&A: {qa_pairs}")

    analyzed_examples.append({
        'original_caption': example['caption'],
        'image_url': example['url'],
        'generated_qa': qa_pairs,
    })
    print("---")

# Create and push dataset
analyzed_dataset = Dataset.from_list(analyzed_examples)
print(analyzed_examples)
if OUTPUT_REPO is not None:
  analyzed_dataset.push_to_hub(OUTPUT_REPO)
  print(f"Dataset available at: {OUTPUT_REPO}")

Streaming LAION dataset...


Resolving data files:   0%|          | 0/128 [00:00<?, ?it/s]


Example 1:
Original caption: """Brent Payne """"Brent Payne"""" 1999 Self Released Country Nm/Nm Out Of Print Cd"""
Image URL: https://www.picclickimg.com/d/l400/pict/333262346250_/Brent-Payne-Brent-Payne-1999-Self-Released-Country.jpg
Generated Q&A: Error: cannot identify image file <_io.BytesIO object at 0x790e8e165b70>
---

Example 2:
Original caption: Universal Orlando 2012 The Ultimate Guide to the Ultimate Theme Park Adventure
Image URL: https://nationalbookswap.com/pbs/m/42/0942/9781887140942.jpg
Generated Q&A: Error: HTTPSConnectionPool(host='nationalbookswap.com', port=443): Max retries exceeded with url: /pbs/m/42/0942/9781887140942.jpg (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))
---

Example 3:
Original caption: Unique 14k Gold Yellow and Blue Diamond Engagement Ring 2.64ct.
Image URL: https://d1251d0o0760fi.cloudfront.net/catalog/product/1/4/14k-gold-di

### 4.2.5 Preparing the dataset for consumption
Getting data from a webdataset sample. Here you can see how we stream data from inside a tar file from a webdataset


In [ ]:


base_url = "https://huggingface.co/datasets/vlmbook/small-publaynet-wds/resolve/main/publaynet-train-{i:06d}.tar"
urls = [base_url.format(i=i) for i in range(4)]
dataset = load_dataset("webdataset", data_files={"train": urls}, split="train", streaming=True)
# From here, use like any HF dataset
for sample in dataset:
    print(f"Sample {sample['__key__']}")
    img_value = sample['png']
    print(f"Annotations: {sample['json']}")
    break


Sample PMC4991227_00003
Annotations: {'annotations': [{'area': 11114.139543321944, 'bbox': [313.74, 51.71, 235.78, 50.59], 'category_id': 1, 'category_name': 'text', 'id': 1690135, 'image_id': 174179, 'iscrowd': 0, 'segmentation': [[313.74, 51.71, 549.51, 51.71, 549.51, 64.75, 549.52, 64.75, 549.52, 76.34, 549.45, 76.34, 549.45, 89.32, 486.89, 89.32, 486.89, 102.3, 313.74, 102.3, 313.74, 90.7, 313.74, 77.73, 313.74, 64.75, 313.74, 51.71]]}, {'area': 36452.22784148902, 'bbox': [59.98, 363.82, 235.83, 154.6], 'category_id': 1, 'category_name': 'text', 'id': 1690136, 'image_id': 174179, 'iscrowd': 0, 'segmentation': [[59.98, 363.82, 295.8, 363.82, 295.8, 375.41, 295.74, 375.41, 295.74, 389.67, 295.75, 389.67, 295.75, 401.43, 295.75, 401.43, 295.75, 415.86, 295.75, 415.86, 295.75, 428.84, 295.81, 428.84, 295.81, 440.43, 295.75, 440.43, 295.75, 454.86, 295.77, 454.86, 295.77, 466.45, 295.74, 466.45, 295.74, 480.81, 295.78, 480.81, 295.78, 492.41, 295.77, 492.41, 295.77, 505.38, 295.75, 505.